In [1]:
import sys, os, time, pickle, warnings

_HERE         = os.path.abspath(os.getcwd())
_PROJECT_ROOT = os.path.dirname(os.path.dirname(_HERE))
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

warnings.filterwarnings("ignore", category=UserWarning)

from utils.seed import fix_all_seeds
fix_all_seeds()

import pandas as pd
import numpy as np
import torch
import requests
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

from config import *
from utils.logger import log_round

CLIENT_ID = "hospital_2"
DATA_FILE = "Final_Depression2_anonymised.csv"

MODEL_FILE           = os.path.join(RESULTS_PATH, f"{CLIENT_ID}_local_model.pth")
GLOBAL_GRADIENT_FILE = AGGREGATED_GRAD_FILE_RAW

print(f"[CONFIG] Client     : {CLIENT_ID}")
print(f"[CONFIG] Experiment : {EXPERIMENT_NAME}")
print(f"[CONFIG] DP enabled : {DP_ENABLED}")
print(f"[CONFIG] HE enabled : {HE_ENABLED}")
print(f"[CONFIG] Seed       : {SEED}")
print(f"[CONFIG] Project root: {BASE_PATH}")


[SEED] All seeds fixed to 456
[CONFIG] Client     : hospital_2
[CONFIG] Experiment : fl_anon_only
[CONFIG] DP enabled : False
[CONFIG] HE enabled : False
[CONFIG] Seed       : 456
[CONFIG] Project root: /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare


In [2]:
data_path = os.path.join(DATA_PATH, DATA_FILE)
df = pd.read_csv(data_path)
print(f"[DATA] Loaded: {data_path}")
print(f"[DATA] Shape:  {df.shape}")
df.head()


[DATA] Loaded: /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/data/anonymized/Final_Depression2_anonymised.csv
[DATA] Shape:  (11696, 39)


,education,urban,gender,engnat,screensize,uniquenetworklocation,hand,religion,orientation,race,...,TIPI2,TIPI3,TIPI4,TIPI5,TIPI6,TIPI7,TIPI8,TIPI9,TIPI10,Condition
0,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_f1f2b172,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_1b5799db,TKN_25a81701,...,2,1,7,4,5,7,7,1,1,Severe
1,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_f1f2b172,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_6b76bbae,TKN_25a81701,...,6,7,5,2,4,6,1,3,3,Mild
2,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_ea75e325,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_1b5799db,TKN_25a81701,...,4,4,5,6,3,7,5,3,1,Moderate
3,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_ea75e325,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_1b5799db,TKN_25a81701,...,4,5,5,5,5,7,5,3,1,Moderate
4,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_ea75e325,TKN_9134e5db,TKN_130c5b34,TKN_05b81f76,TKN_2c94cae9,TKN_25a81701,...,5,4,5,5,4,6,5,1,2,Extremely Severe


In [3]:
target_column = 'Condition'
df_encoded = df.copy()
label_encoders = {}

for col in df_encoded.select_dtypes(include='object').columns:
    if col != target_column:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
        label_encoders[col] = le

le_target = LabelEncoder()
df_encoded[target_column] = le_target.fit_transform(df_encoded[target_column])

X = df_encoded.drop(columns=[target_column])
y = df_encoded[target_column]

print(f"[PREP] Features: {X.shape[1]}")
print(f"[PREP] Samples:  {X.shape[0]}")
print(f"[PREP] Classes:  {list(le_target.classes_)}")


[PREP] Features: 38
[PREP] Samples:  11696
[PREP] Classes:  ['Extremely Severe', 'Mild', 'Moderate', 'Normal', 'Severe']


In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=SEED
)
print(f"[SPLIT] Train: {X_train.shape[0]}  Val: {X_val.shape[0]}")


[SPLIT] Train: 9356  Val: 2340


In [5]:
smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"[SMOTE] Before: {X_train.shape[0]}  After: {X_train_sm.shape[0]}")


[SMOTE] Before: 9356  After: 15625


In [6]:
X_train_tensor = torch.tensor(X_train_sm.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_sm,         dtype=torch.long)
X_val_tensor   = torch.tensor(X_val.values,       dtype=torch.float32)
y_val_tensor   = torch.tensor(y_val.values,       dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor,   y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f"[DATA] Train batches: {len(train_loader)}")
print(f"[DATA] Val batches:   {len(val_loader)}")


[DATA] Train batches: 489
[DATA] Val batches:   74


In [7]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, HIDDEN_DIM_1),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM_1, HIDDEN_DIM_2),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM_2, num_classes)
        )
    def forward(self, x):
        return self.model(x)

input_dim   = X_train_sm.shape[1]
num_classes = OUTPUT_DIM
model       = MLP(input_dim, num_classes)

total_params = sum(p.numel() for p in model.parameters())
print(f"[MODEL] Architecture: {input_dim} → "
      f"{HIDDEN_DIM_1} → {HIDDEN_DIM_2} → {num_classes}")
print(f"[MODEL] Total parameters: {total_params}")


[MODEL] Architecture: 38 → 128 → 64 → 5
[MODEL] Total parameters: 13573


In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
print("[DP] Differential privacy DISABLED for this variant")


[DP] Differential privacy DISABLED for this variant


In [9]:
best_val_loss    = float('inf')
patience_counter = 0

for epoch in range(MAX_EPOCHS):
    start = time.time()

    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            out = model(X_batch)
            val_loss += criterion(out, y_batch).item()
            all_preds.extend(torch.argmax(out, dim=1).numpy())
            all_labels.extend(y_batch.numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    elapsed = time.time() - start

    log_round(
        client_id=CLIENT_ID,
        fl_round=epoch,
        val_accuracy=round(val_acc, 4),
        train_loss=round(train_loss, 4),
        val_loss=round(val_loss, 4),
        epsilon=None,
        elapsed_seconds=round(elapsed, 2)
    )

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_FILE)
        print(f"  ✓ Best model saved → {MODEL_FILE}")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"[STOP] Early stopping at epoch {epoch+1}")
            break


[LOG] Saved → results/fl_anon_only.jsonl
Epoch 01 | Train Loss: 318.4814 | Val Loss: 35.9292 | Val Acc: 0.7846
  ✓ Best model saved → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/hospital_2_local_model.pth
[LOG] Saved → results/fl_anon_only.jsonl
Epoch 02 | Train Loss: 215.7983 | Val Loss: 30.4261 | Val Acc: 0.8158
  ✓ Best model saved → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/hospital_2_local_model.pth
[LOG] Saved → results/fl_anon_only.jsonl
Epoch 03 | Train Loss: 196.8305 | Val Loss: 29.4460 | Val Acc: 0.8171
  ✓ Best model saved → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/hospital_2_local_model.pth
[LOG] Saved → results/fl_anon_only.jsonl
Epoch 04 | Train Loss: 190.8499 | Val Loss: 31.7159 | Val Acc: 0.8222
[LOG] Saved → results/fl_anon_only.jsonl
Epoch 05 | Train Loss: 182.1778 | Val Loss: 28.6670 | Val Acc: 0.8350
  ✓ Best model saved → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/hospital_2_local_mod

In [10]:
def flatten_gradients(mdl):
    grads = [p.grad.view(-1).cpu()
             for p in mdl.parameters() if p.grad is not None]
    flat = torch.cat(grads)
    print(f"[GRAD] Flattened gradient length: {len(flat)}")
    return flat

def send_raw_gradients(flat_grads, url, client_id):
    chunks = torch.chunk(flat_grads,
                         (len(flat_grads) + CHUNK_SIZE - 1) // CHUNK_SIZE)
    for i, chunk in enumerate(chunks):
        payload = pickle.dumps({
            'chunk_id':  i,
            'client_id': client_id,
            'data':      chunk.numpy()
        })
        try:
            r = requests.post(
                url, data=payload,
                headers={'Content-Type': 'application/octet-stream'},
                verify=False
            )
            status = "OK" if r.status_code == 200 else f"HTTP {r.status_code}"
            print(f"[GRAD] Sent chunk {i} ({len(payload)} bytes) → {status}")
        except Exception as e:
            print(f"[GRAD] Error sending chunk {i}: {e}")

def receive_and_apply_gradient(mdl, grad_file):
    with open(grad_file, "rb") as f:
        aggregated_chunks = pickle.load(f)

    global_grad = torch.cat([
        torch.tensor(chunk, dtype=torch.float32)
        for chunk in aggregated_chunks
    ])
    print(f"[GRAD] Total gradient length: {len(global_grad)}")

    norm = global_grad.norm()
    if norm > 1e2:
        global_grad = global_grad / norm * 1e2
        print(f"[GRAD] Gradient clipped (original norm: {norm:.2e})")

    ptr = 0
    for param in mdl.parameters():
        if param.grad is not None:
            sz = param.grad.numel()
            param.data -= global_grad[ptr:ptr+sz].view(param.grad.shape)
            ptr += sz
    print("[GRAD] Global gradient applied to model")

def evaluate(mdl, loader, crit, split="Val"):
    mdl.eval()
    total_loss, preds, labels = 0.0, [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = mdl(xb)
            total_loss += crit(out, yb).item()
            preds.extend(torch.argmax(out, dim=1).numpy())
            labels.extend(yb.numpy())
    acc = accuracy_score(labels, preds)
    print(f"[EVAL] {split} → Acc: {acc:.4f}  Loss: {total_loss:.4f}")
    return acc, total_loss, preds, labels


In [ ]:
global_model = MLP(input_dim, num_classes)
state_dict   = torch.load(MODEL_FILE)
fixed_sd     = {k.replace("_module.", ""): v for k, v in state_dict.items()}
global_model.load_state_dict(fixed_sd, strict=False)
print(f"[FL] Loaded model from {MODEL_FILE}")

fl_criterion = nn.CrossEntropyLoss()
fl_optimizer = torch.optim.Adam(global_model.parameters(), lr=LEARNING_RATE)

best_val_loss    = float('inf')
patience_counter = 0

for fl_round in range(FL_ROUNDS):
    print(f"\n{'='*50}")
    print(f"  Federated Round {fl_round + 1} / {FL_ROUNDS}")
    print(f"{'='*50}")
    round_start = time.time()

    # STEP 1: local training pass
    global_model.train()
    for xb, yb in train_loader:
        fl_optimizer.zero_grad()
        out  = global_model(xb)
        loss = fl_criterion(out, yb)
        loss.backward()
        fl_optimizer.step()

    # STEP 2: send raw gradients
    grads = flatten_gradients(global_model)
    send_raw_gradients(grads, SERVER_URL, CLIENT_ID)

    # STEP 3: poll for aggregated file
    print(f"[FL] Waiting for aggregated gradient file...")
    wait_start = time.time()
    while not os.path.exists(GLOBAL_GRADIENT_FILE):
        time.sleep(2)
        if time.time() - wait_start > 300:
            print("[FL] Timeout waiting for server. Skipping round.")
            break

    # STEP 4: apply global gradient
    receive_and_apply_gradient(global_model, GLOBAL_GRADIENT_FILE)

    # STEP 5: evaluate
    val_acc, val_loss, val_preds, val_labels = evaluate(
        global_model, val_loader, fl_criterion, "Val"
    )
    train_acc, train_loss, _, _ = evaluate(
        global_model, train_loader, fl_criterion, "Train"
    )

    elapsed = time.time() - round_start
    print(f"[FL] Round {fl_round+1} complete in {elapsed:.1f}s")
    print(classification_report(val_labels, val_preds, zero_division=0))

    log_round(
        client_id=CLIENT_ID,
        fl_round=fl_round + 1,
        val_accuracy=round(val_acc, 4),
        train_loss=round(train_loss, 4),
        val_loss=round(val_loss, 4),
        epsilon=None,
        elapsed_seconds=round(elapsed, 2)
    )

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        save_path = os.path.join(RESULTS_PATH, f"{CLIENT_ID}_global_model.pth")
        torch.save(global_model.state_dict(), save_path)
        print(f"[FL] ✓ Best global model saved → {save_path}")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"[FL] Early stopping at round {fl_round+1}")
            break

print(f"\n[FL] Training complete. Best val loss: {best_val_loss:.4f}")


[FL] Loaded model from /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/hospital_2_local_model.pth

  Federated Round 1 / 10
[GRAD] Flattened gradient length: 13573


/Users/ravi/anaconda3/envs/privfedhealth/lib/python3.10/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[GRAD] Sent chunk 0 (27348 bytes) → OK
[GRAD] Sent chunk 1 (27344 bytes) → OK
[FL] Waiting for aggregated gradient file...


/Users/ravi/anaconda3/envs/privfedhealth/lib/python3.10/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
